# Dark QC

Checks every dark of one night against a reference dark stack (made by `calib_stack.ipynb`), the same way as
`bias_qc.ipynb` checks the zeros. Every dark is first made comparable with the stack: minus the bias `mean` map that
the stack was made with, divided by the dark's own EXPTIME, in ADU/s. Then z = (dark - ref dark mean) / noise, where
noise is the expected scatter of that pixel in one dark: the read noise (bias `std`) and the dark noise (`std_excess` x
EXPTIME) added in squares, divided by EXPTIME. For a good dark z is pure noise with sigma ~1. Darks of any exposure time
are checked (each with its own EXPTIME).

1. **offset** [ADU/s]: median of (dark - ref dark mean), the change of the dark current of the whole frame (the jump
   of the bias level, ~0.75 ADU on wk06, gives ~0.001 ADU/s at 900 s).
2. **zsig**: sigma of z from its MAD; > 1 means the dark is noisier than usual.
3. **tails**: number of pixels with z more than 5 above / below the median of z. Pure noise gives ~1 on each side.
4. **cosmics**: number of groups of touching pixels with z above 10 above the median of z (many in a long dark).

EXPTIME, T-CAM, the position in the series (LOOP), dome and mirror cover are only listed, not judged.

Darks are rare (on wk06 a few nights a month), so the reference is set by hand: a block of 10 nights with a dark stack.
If it contains the night itself, the checked darks are part of the reference.

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
from scipy import ndimage
from tqdm.notebook import tqdm

TEL = 'wk06'
NIGHT = 1332        # night to check, int(JD) - 2460000
REF_START = 1330    # reference dark stack, set by hand
REF_STOP = 1340

RAW_DIR = f'/data/fits/{TEL}/raw'
BIAS_DIR = os.path.expanduser(f'~/data/ocm_bias/{TEL}')
DARK_DIR = os.path.expanduser(f'~/data/ocm_dark/{TEL}')

ref_dark = fits.getdata(f'{DARK_DIR}/{TEL}_{REF_START}_{REF_STOP}_mean.fits')
ref_excess = fits.getdata(f'{DARK_DIR}/{TEL}_{REF_START}_{REF_STOP}_std_excess.fits')
h = fits.getheader(f'{DARK_DIR}/{TEL}_{REF_START}_{REF_STOP}_mean.fits')
bias_name = h['BIAS']  # the bias stack the reference dark was made with
ref_bias = fits.getdata(f'{BIAS_DIR}/{bias_name}_mean.fits')
ref_bias_std = fits.getdata(f'{BIAS_DIR}/{bias_name}_std.fits')
print(f"reference {TEL}_{REF_START}_{REF_STOP}: {h['NDARK']} darks of {h['EXPTIME']:.0f} s, dark {np.median(ref_dark):.5f} ADU/s, bias {bias_name}")

reference wk06_1330_1340: 12 darks of 900 s, dark 0.00499 ADU/s, bias wk06_1330_1340


## Find the darks

In [ ]:
# old: reading every header (replaced by the frames index in the next cell)
# dark_list = []
# for f in tqdm(sorted(glob.glob(f'{RAW_DIR}/{NIGHT}/*.fits')), desc=f'night {NIGHT}'):
#     h = fits.getheader(f)
#     if h['IMAGETYP'] == 'dark':
#         dark_list.append(f)
#
# print(len(dark_list), 'darks')

In [ ]:
# the frames of these nights from the index made by frames_index.ipynb
import pandas as pd
FRAMES_DIR = os.path.expanduser(f'~/data/ocm_frames/{TEL}')
nights = [n for n in [NIGHT] if os.path.exists(f'{FRAMES_DIR}/{n}.parquet')]
missing = [n for n in [NIGHT] if n not in nights]
if missing:
    print('nights not in the index (no frames, or run frames_index.ipynb):', missing)
frames = pd.concat([pd.read_parquet(f'{FRAMES_DIR}/{n}.parquet') for n in nights], ignore_index=True)

dark_list = list(frames.loc[frames['IMAGETYP'] == 'dark', 'file'])

print(len(dark_list), 'darks')

## Check every dark

One row per dark. `n_pos` / `n_neg`: pixels more than 5 above / below; `cosmics`: groups of touching pixels above 10
(`ndimage.label` numbers the groups of touching True pixels, the number of groups is what we count).

In [3]:
rows = []
for f in tqdm(dark_list, desc='darks'):
    with fits.open(f) as hdul:
        h = hdul[0].header
        data = hdul[0].data.astype(np.float32)
    t = h['EXPTIME']

    rate = (data - ref_bias) / t                                       # ADU/s
    noise = np.sqrt(ref_bias_std ** 2 + (ref_excess * t) ** 2) / t     # expected scatter of one dark, ADU/s
    diff = rate - ref_dark
    z = diff / noise
    z = z - np.median(z)  # without the shift of the whole frame (that is the offset)

    rows.append({
        'file': os.path.basename(f),
        'time': h['DATE-OBS'][11:19],
        'exptime': t,
        'loop': h.get('LOOP'),
        'tcam': h.get('T-CAM'),
        'dome': h.get('DOME-SHT'),
        'mirror': h.get('MIRR-COV'),
        'offset': np.median(diff),
        'zsig': 1.4826 * np.median(np.abs(z)),
        'n_pos': (z > 5).sum(),
        'n_neg': (z < -5).sum(),
        'cosmics': ndimage.label(z > 10)[1],
    })

qc = pd.DataFrame(rows)
pd.set_option('display.max_rows', 500)
qc.round(5)

darks:   0%|          | 0/18 [00:00<?, ?it/s]

,file,time,exptime,loop,tcam,dome,mirror,offset,zsig,n_pos,n_neg,cosmics
0,wk06c_1332_47190.fits,23:19:32,1800.0,1,-59.128,1,1,-0.00028,0.95049,4296,63,760
1,wk06c_1332_49281.fits,23:49:38,1800.0,2,-60.426,1,1,0.00035,1.05333,4624,132,762
2,wk06c_1332_51373.fits,00:19:45,1800.0,3,-60.426,1,1,-0.00049,1.06614,4394,191,728
3,wk06c_1332_53463.fits,00:49:52,1800.0,4,-59.128,1,1,-0.00090,0.95879,4176,67,740
4,wk06c_1332_55555.fits,01:19:59,1800.0,5,-60.426,1,1,-0.00045,0.96610,4897,66,819
5,wk06c_1332_57647.fits,01:50:07,1800.0,6,-60.426,1,1,-0.00036,1.05056,4342,352,724
6,wk06c_1332_59912.fits,02:22:44,900.0,1,-60.426,1,1,0.00167,0.96042,1910,3,356
7,wk06c_1332_60962.fits,02:37:51,900.0,2,-60.426,1,1,0.00250,1.03470,1990,3,353
8,wk06c_1332_62012.fits,02:52:58,900.0,3,-59.777,1,1,-0.00065,1.03933,2042,9,396
9,wk06c_1332_63062.fits,03:08:05,900.0,4,-59.128,1,1,-0.00000,0.94741,2083,2,390


## One dark

The z map of one dark: by default the noisiest one (largest `zsig`); put a file name from the table to see another.
The histogram has the Gaussian of pure noise (sigma 1) for comparison.

In [ ]:
#f = qc.loc[qc['zsig'].idxmax(), 'file']   
#f = 'wk06c_1332_71461.fits'
f = os.path.basename(dark_list[0])

data = fits.getdata(f'{RAW_DIR}/{NIGHT}/{f}').astype(np.float32)
t = fits.getheader(f'{RAW_DIR}/{NIGHT}/{f}')['EXPTIME']
rate = (data - ref_bias) / t
noise = np.sqrt(ref_bias_std ** 2 + (ref_excess * t) ** 2) / t
z = (rate - ref_dark) / noise
z = z - np.median(z)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={'width_ratios': [1, 1.2]})
fig.suptitle(f'{f} ({t:.0f} s):  zsig {1.4826 * np.median(np.abs(z)):.3f}   {(z > 5).sum()} above 5   {(z < -5).sum()} below -5', fontsize=13)
im = ax1.imshow(z, origin='lower', vmin=-5, vmax=5, cmap='RdBu_r')
fig.colorbar(im, ax=ax1, shrink=0.8, label='z')

v = z.ravel()
counts, edges, _ = ax2.hist(v, bins=np.arange(-20, 20.1, 0.2), histtype='step', color='k')
width = edges[1] - edges[0]
x = np.linspace(-20, 20, 2000)
ax2.plot(x, len(v) * width / np.sqrt(2 * np.pi) * np.exp(-0.5 * x ** 2), 'r')
ax2.set_yscale('log')
ax2.set_ylim(0.5, None)
ax2.set_xlabel('z')
plt.tight_layout()
plt.show()